<a href="https://colab.research.google.com/github/juancatala/03MAIR---Algoritmos-de-Optimizacion---2025/blob/main/Trabajo%20Final/Edicion_OCT25_Trabajo_Pr%C3%A1ctico_Algoritmos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Juan Alfonso Catalá Aparisi  <br>
Url: https://colab.research.google.com/drive/1pD5hbMpHjcRlCz5X5X48eYB9F1lnRyi3?usp=sharing <br>
Google Colab: https://github.com/juancatala/03MAIR---Algoritmos-de-Optimizacion---2025 <br>
Problema:
>1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de una jornada de La Liga<br>
>3. Configuración de Tribunales

Descripción del problema:

##Sesiones de doblaje

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:

Número de actores: 10

Número de tomas : 30

Actores/Tomas : https://bit.ly/36D8IuK
- 1 indica que el actor participa en la toma
- 0 en caso contrario








                                        

#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

In [ ]:
#Respuesta


## Modelo


- ¿Cómo represento el espacio de soluciones?

Para organizar el plan de grabación he utilizado una lista de listas, que en mi código se llama "plan". Cada una de las listas internas representa un día de trabajo, y dentro de ellas guardo los números que identifican a cada toma. Me parece la forma más limpia de trabajar porque me permite mover tomas de un día a otro muy fácilmente.


- ¿Cuál es la función objetivo?

Mi objetivo es que el doblaje de la película cueste lo menos posible, y como el gasto viene de los actores que van al estudio cada día, lo que busco es que el número total de visitas de actores sea el mínimo. En mi función "coste_plan", lo que hago es mirar qué actores participan en las tomas de cada día, cuento cuántos hay sin repetir y sumo esos totales de todos los días. Cuanto más pequeña sea esa suma, mejor es mi solución.


- ¿Cómo implemento las restricciones?

La regla de oro es que no puede haber más de 6 tomas por día. En lugar de dejar que el algoritmo cree cualquier plan y luego castigarlo si se pasa, he preferido que sea imposible romper la regla. Cuando el código intenta mover una toma a un día nuevo, lo primero que hace es comprobar si ese día ya tiene 6 tomas, y si está lleno entonces el movimiento se cancela y busca otro. Así me aseguro de que el plan siempre sea válido.

#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

In [ ]:
#Respuesta


## Análisis


- ¿Qué complejidad tiene el problema?

Este problema es de una complejidad de orden exponencial, y su dificultad es considerable, lo que se conoce como NP-Duro. No se puede resolver de forma perfecta simplemente siguiendo unos pasos fijos en poco tiempo, ya que a medida que añades más tomas o más actores el esfuerzo que necesita el ordenador para encontrar la mejor combinación crece de forma explosiva. No hay un atajo matemático que nos dé la solución ideal directamente.


- Contabilizar el espacio de soluciones

El número de combinaciones posibles es astronómico. Con 30 tomas, hay muchísimas formas de repartirlas en grupos de seis. Si intentáramos probar todas las combinaciones una por una para ver cuál es la más barata el ordenador tardaría demasiado tiempo, como con el Backtracking. Por eso es totalmente necesario usar una técnica como el recocido simulado, que explora de forma inteligente sin tener que mirarlo todo.

#Diseño
- ¿Que técnica utilizo? ¿Por qué?

In [ ]:
#Respuesta

## Diseño


- ¿Qué técnica utilizo?

Algoritmo Metaheurístico: Recocido Simulado (Simulated Annealing).


- ¿Por qué?

La elegí principalmente porque otros métodos más sencillos suelen quedarse atascados, como con el Hill Climbing. Si solo aceptara cambios que mejoran el coste, llegaría un punto en el que no vería ninguna mejora obvia y se pararía, aunque hubiera una solución mucho mejor un poco más allá.


El recocido simulado tiene un truco muy interesante. Al principio, cuando la "temperatura" es alta, se permite el lujo de aceptar cambios que empeoran un poco el coste. Esto suena raro, pero es lo que le permite salir de soluciones que parecen buenas pero no lo son tanto, para seguir explorando. Conforme avanza el tiempo y la temperatura baja, el algoritmo se vuelve más selectivo con su toma de decisiones y se queda con lo mejor que haya encontrado.


Además, para ayudar al algoritmo, no empecé de cero. Hice que el programa fuera un poco listo al principio ordenando las tomas que tienen más actores para intentar agruparlas cuanto antes. Esto le da un buen empujón inicial al proceso.

In [ ]:
# Importar librerias
import random
import math
import copy

In [ ]:
# -------------------------
# Inicializacion
# -------------------------

# Matriz de doblaje
matriz = [
    [1,1,1,1,1,0,0,0,0,0],
    [0,0,1,1,1,0,0,0,0,0],
    [0,1,0,0,1,0,1,0,0,0],
    [1,1,0,0,0,0,1,1,0,0],
    [0,1,0,1,0,0,0,1,0,0],
    [1,1,0,1,1,0,0,0,0,0],
    [1,1,0,1,1,0,0,0,0,0],
    [1,1,0,0,0,1,0,0,0,0],
    [1,1,0,1,0,0,0,0,0,0],
    [1,1,0,0,0,1,0,0,1,0],
    [1,1,1,0,1,0,0,1,0,0],
    [1,1,1,1,0,1,0,0,0,0],
    [1,0,0,1,1,0,0,0,0,0],
    [1,0,1,0,0,1,0,0,0,0],
    [1,1,0,0,0,0,1,0,0,0],
    [0,0,0,1,0,0,0,0,0,1],
    [1,0,1,0,0,0,0,0,0,0],
    [0,0,1,0,0,1,0,0,0,0],
    [1,0,1,0,0,0,0,0,0,0],
    [1,0,1,1,1,0,0,0,0,0],
    [0,0,0,0,0,1,0,1,0,0],
    [1,1,1,1,0,0,0,0,0,0],
    [1,0,1,0,0,0,0,0,0,0],
    [0,0,1,0,0,1,0,0,0,0],
    [1,1,0,1,0,0,0,0,0,1],
    [1,0,1,0,1,0,0,0,1,0],
    [0,0,0,1,1,0,0,0,0,0],
    [1,0,0,1,0,0,0,0,0,0],
    [1,0,0,0,1,1,0,0,0,0],
    [1,0,0,1,0,0,0,0,0,0]
]

# Semilla para poder reproducir los resultados
random.seed(666)

# Maximo numero de tomas por dia
max_tomas = 6

# Generamos sets de actores por toma
actores_por_toma = [set(i for i, v in enumerate(t) if v == 1) for t in matriz]

# Colocar primero las tomas con mas actores
tomas = list(range(len(matriz)))
tomas.sort(key=lambda t: len(actores_por_toma[t]), reverse=True)

# Devolver el set de actores que trabaja ese dia
def actores_dia(tomas):
    res = set()
    for t in tomas:
        res.update(actores_por_toma[t])
    return res

# Devolver el sumatiorio de actores del dia
def coste_plan(plan):
    return sum(len(actores_dia(d)) for d in plan)

# Contruccion secuencial de las tomas
plan = [tomas[i:i + max_tomas]for i in range(0, len(tomas), max_tomas)]
coste = coste_plan(plan)

# Guardar la mejor solucion
mejor = copy.deepcopy(plan)
mejor_coste = coste


# -------------------------
# Recocido simulado
# -------------------------

temp = coste * 10 # temperatura inicial
iter_temp = 100 # numero de vecinos por nivel
enfriamiento = 0.995 # factor de reduccion de temp

while temp > 0.1:
    for _ in range(iter_temp):
        # Se aplica el movimiento sobre 2 dias distintos
        i1, i2 = random.sample(range(len(plan)), 2)
        d1, d2 = plan[i1], plan[i2]

        # Coste de los dias antes del cambio
        coste_prev = len(actores_dia(d1)) + len(actores_dia(d2))

        # Cambiar toma o dia
        mov = random.choice(['mover', 'swap'])

        # Mover la toma de un dia al otro
        if mov == 'mover':
            origen, destino = random.choice([(d1, d2), (d2, d1)])
            if origen and len(destino) < max_tomas:
                t = random.choice(origen)
                origen.remove(t)
                destino.append(t)

                # Recalcular el coste
                coste_new = len(actores_dia(d1)) + len(actores_dia(d2))
                diff = coste_new - coste_prev

                # Criterio de aceptacion
                if diff < 0 or random.random() < math.exp(-diff / temp):
                    coste += diff

                    # Actualizar coste y plan si la solucion es mejor
                    if coste < mejor_coste:
                        mejor_coste = coste
                        mejor = copy.deepcopy(plan)

                # Si la solucion no es mejor, deshacer movimiento
                else:
                    destino.remove(t)
                    origen.append(t)

        # Cambiar tomas entre los 2 dias
        elif mov == 'swap' and d1 and d2:
            a, b = random.randrange(len(d1)), random.randrange(len(d2))
            # Guardar cambio
            t1, t2 = d1[a], d2[b]
            d1[a], d2[b] = t2, t1

            # Recalcular el coste
            coste_new = len(actores_dia(d1)) + len(actores_dia(d2))
            diff = coste_new - coste_prev

            # Criterio de aceptacion
            if diff < 0 or random.random() < math.exp(-diff / temp):
                coste += diff

                # Actualizar coste y plan si la solucion es mejor
                if coste < mejor_coste:
                    mejor_coste = coste
                    mejor = copy.deepcopy(plan)

            # Si la solucion no es mejor, deshacer movimiento
            else:
                d1[a], d2[b] = t1, t2

    # Reducir temperatura tras la exploracion
    temp *= enfriamiento


# -------------------------
# Resultados
# -------------------------

print(f"Valor función objetivo: {mejor_coste}")
print("Plan de doblaje:")

# Ordenar resultados
for i, d in enumerate(mejor):
    if d:
        act = sorted(a + 1 for a in actores_dia(d)) # +1 porque el primer indice es 0
        ts = sorted(t + 1 for t in d)
        print(f"\tDía {i+1} -> Tomas {ts} | Actores {act}")

Valor función objetivo: 27
Plan de doblaje:
	Día 1 -> Tomas [3, 4, 8, 15, 21, 29] | Actores [1, 2, 5, 6, 7, 8]
	Día 2 -> Tomas [2, 5, 10, 11, 12, 26] | Actores [1, 2, 3, 4, 5, 6, 8, 9]
	Día 3 -> Tomas [1, 13, 16, 20, 22, 25] | Actores [1, 2, 3, 4, 5, 10]
	Día 4 -> Tomas [14, 17, 18, 19, 23, 24] | Actores [1, 3, 6]
	Día 5 -> Tomas [6, 7, 9, 27, 28, 30] | Actores [1, 2, 4, 5]
